# Lezione 09 — File: Leggere e Scrivere Dati

## Obiettivi della lezione

Al termine di questa lezione sarai in grado di:

- Aprire file con `open()` comprendendo le modalità `'r'`, `'w'`, `'a'` e `'x'`.
- Usare `with` come modo sicuro e standard di lavorare con i file.
- Leggere contenuti con `read()`, `readline()` e `readlines()`, e sapere quando usare ciascuno.
- Scrivere e aggiungere contenuti con `write()`.
- Copiare o trasformare il contenuto di un file verso un altro.
- Controllare il cursore di un file con `seek()` e `tell()`, e usare le modalità combinate `r+`, `w+`, `a+`.

## Perché ti dovrebbe interessare?

Finora i dati vivevano dentro il tuo codice: li avevi scritti tu in una lista o variabile. Nella vita reale, i dati arrivano in file — un `.csv` con le vendite, un `.txt` con dei log, un `.json` di un'API salvato su disco. Prima di ripulire, analizzare o graficare qualsiasi cosa, bisogna sapere come estrarre quelle informazioni da un file e portarle dentro Python (e, viceversa, salvare i risultati). `pandas` e `numpy` fanno esattamente questo sotto il cofano, con comodità aggiuntive sopra — capire cosa c'è sotto evita molti mal di testa più avanti.

## 1. `open()` e le sue modalità: la porta d'ingresso a qualsiasi file

`open()` riceve due cose: il percorso del file e la **modalità**. Se non indichi la modalità, Python assume `'r'`.

| Modalità | Cosa fa |
|---|---|
| `'r'` | Legge un file esistente. Se non esiste, lancia un errore. |
| `'w'` | Scrive. Crea il file se non esiste, e **cancella tutto il contenuto** se già esisteva. |
| `'a'` | Aggiunge (append). Scrive alla fine senza cancellare nulla. Crea il file se non esiste. |
| `'x'` | Creazione esclusiva. Crea il file, ma lancia un errore se esiste già. |

L'errore più comune di chi inizia: confondere `'w'` con `'a'`. Aprire in `'w'` per "aggiungere qualcosa" in realtà svuota prima il file.

In [ ]:
# 'w' sovrascrive: se 'notas.txt' aveva già del contenuto, sparisce
with open('notas.txt', 'w') as archivo:
    archivo.write("Questa è l'unica cosa che rimarrà\n")

with open('notas.txt', 'r') as archivo:
    print(archivo.read())

In [ ]:
# 'a' rispetta ciò che c'era già e aggiunge dopo
with open('notas.txt', 'a') as archivo:
    archivo.write("Questo viene aggiunto alla fine, senza cancellare nulla\n")

with open('notas.txt', 'r') as archivo:
    print(archivo.read())

## 2. `with`: perché non dovresti aprire un file "a mano"

Quando apri un file con `open()`, Python ti dà un **oggetto file** — come una chiave prestata dal sistema operativo. Se non la restituisci (`close()`), quelle risorse restano bloccate, e se il programma va in crash a causa di un'eccezione a metà strada, il file non viene mai chiuso correttamente.

In [ ]:
# Forma manuale: funziona, ma è rischiosa
archivo = open('notas.txt', 'r')
contenido = archivo.read()
archivo.close()  # se qualcosa fallisce PRIMA di questa riga, il file non viene mai chiuso
print(contenido)

In [ ]:
# Forma corretta e consigliata: with agisce come context manager
with open('notas.txt', 'r') as archivo:
    contenido = archivo.read()
# Uscendo dal blocco, il file si chiude da solo, qualunque cosa succeda dentro

print(archivo.closed)  # True -> with lo ha già chiuso per te
print(archivo.name)    # 'notas.txt'
print(archivo.mode)    # 'r'

## 3. Tre modi di leggere: `read()`, `readline()` e `readlines()`

**`read()` — prende tutto in una volta.** Restituisce tutto il contenuto come un'unica stringa, con gli `\n` inclusi.

In [ ]:
# Creiamo un file con diverse righe per gli esempi che seguono
with open('datos.txt', 'w') as archivo:
    archivo.write("prima riga\n")
    archivo.write("seconda riga\n")
    archivo.write("terza riga\n")

with open('datos.txt', 'r') as archivo:
    contenido = archivo.read()
    print(contenido)
    print(type(contenido))

In [ ]:
# read(n) legge i successivi n caratteri da dove si trova il cursore.
# È cumulativo: ogni chiamata continua da dove è arrivata la precedente.
with open('datos.txt', 'r') as archivo:
    print(archivo.read(4))   # i primi 4 caratteri
    print(archivo.read(4))   # i successivi 4 caratteri
    print(archivo.read(7))   # i successivi 7 caratteri

**`readline()` — una riga alla volta.** Ogni chiamata restituisce la riga successiva come stringa.

In [ ]:
with open('datos.txt', 'r') as archivo:
    primera = archivo.readline()
    segunda = archivo.readline()
    print(primera)
    print(segunda)

In [ ]:
# Per leggere tutto il file senza sapere quante righe ha:
# il ciclo si interrompe quando readline() restituisce una stringa vuota (che Python interpreta come False)
with open('datos.txt', 'r') as archivo:
    while True:
        linea = archivo.readline()
        if not linea:
            break  # non restano più righe
        print(linea)

In [ ]:
# Forma più diretta e la più usata nella pratica: iterare direttamente sull'oggetto file
with open('datos.txt', 'r') as archivo:
    for i, linea in enumerate(archivo):
        print("Riga", i, ":", linea)

**`readlines()` — tutte le righe, ma come lista.** Utile quando hai bisogno di accesso indicizzato (la riga 5, la penultima, ecc.).

In [ ]:
with open('datos.txt', 'r') as archivo:
    lista_lineas = archivo.readlines()

print(lista_lineas)
print(lista_lineas[0])  # la prima riga
print(lista_lineas[1])  # la seconda riga

**Riepilogo per decidere:** file piccolo e vuoi tutto il testo in una volta → `read()`. Devi scorrere riga per riga senza caricare tutto in memoria → `for linea in archivo` o `readline()` in un ciclo. Ti servono le righe come lista indicizzabile → `readlines()`.

## 4. Scrivere e aggiungere: `write()`, modalità `'w'` e modalità `'a'`

A differenza di `print()`, `write()` **non aggiunge automaticamente un a capo** — se non metti tu il `\n`, tutto il testo resta attaccato.

In [ ]:
# Creare (o sovrascrivere) un file e scrivere due righe
with open('reporte.txt', 'w') as archivo:
    archivo.write("Questa è la riga A\n")
    archivo.write("Questa è la riga B\n")

with open('reporte.txt', 'r') as archivo:
    print(archivo.read())

In [ ]:
# Quando hai diverse righe in una lista, la cosa naturale è scorrerla con un for
lineas = ["Prima riga", "Seconda riga", "Terza riga"]

with open('reporte.txt', 'w') as archivo:
    for linea in lineas:
        archivo.write(linea + "\n")

with open('reporte.txt', 'r') as archivo:
    print(archivo.read())

Ogni volta che esegui quel blocco con modalità `'w'`, il file viene creato da zero. Se invece vuoi **conservare** ciò che c'è già e solo aggiungere informazioni nuove (ad esempio, un log), usi la modalità `'a'`.

In [ ]:
nuevo_dato = "Questa è la riga C"

with open('reporte.txt', 'a') as archivo:
    archivo.write(nuevo_dato + "\n")

with open('reporte.txt', 'r') as archivo:
    print(archivo.read())

Un pattern molto comune nella data science è copiare o trasformare il contenuto di un file verso un altro: leggi riga per riga dall'origine e scrivi nella destinazione, annidando due `with`. Questo pattern è la base di qualsiasi processo che filtra, ripulisce o riorganizza dati.

In [ ]:
# Prepariamo un file di origine
with open('origen.txt', 'w') as archivo:
    archivo.write("dato 1\n")
    archivo.write("dato 2\n")
    archivo.write("dato 3\n")

# Copiamo il suo contenuto in un file di destinazione, riga per riga
with open('origen.txt', 'r') as archivo_origen:
    with open('destino.txt', 'w') as archivo_destino:
        for linea in archivo_origen:
            archivo_destino.write(linea)

with open('destino.txt', 'r') as archivo:
    print(archivo.read())

## 5. `seek()`, `tell()` e le modalità combinate (`r+`, `w+`, `a+`)

Ogni file aperto ha un **cursore** invisibile che indica da dove si legge o scrive successivamente.

- `archivo.tell()` — posizione attuale del cursore (in byte).
- `archivo.seek(offset, da_dove)` — sposta il cursore. `da_dove`: `0` inizio, `1` posizione attuale, `2` fine.

In [ ]:
with open('reporte.txt', 'r') as archivo:
    print(archivo.tell())        # 0, appena aperto
    archivo.read(10)
    print(archivo.tell())        # 10, avanzato di 10 byte
    archivo.seek(0, 0)           # torna all'inizio
    print(archivo.tell())        # 0 di nuovo

Le modalità combinate permettono di leggere e scrivere sullo stesso file aperto:

- **`r+`** — lettura e scrittura su un file esistente. Non lo svuota, ma non lo crea nemmeno se non esiste.
- **`w+`** — scrittura e lettura, ma **tronca** il file all'apertura (come `'w'` da solo).
- **`a+`** — scrittura (sempre alla fine) e lettura. Crea il file se non esiste.

La trappola con `a+`: anche se puoi leggere, il cursore di scrittura resta sempre alla fine, quindi se chiami `read()` senza prima spostare il cursore, è comune non ottenere nulla.

In [ ]:
with open('reporte.txt', 'a+') as archivo:
    archivo.write("Un'altra riga\n")
    contenido = archivo.read()   # probabilmente vuoto: il cursore è rimasto alla fine
    if not contenido:
        print("Non ha letto nulla perché il cursore è alla fine")

    archivo.seek(0, 0)           # ora sì, torniamo all'inizio
    print(archivo.read())        # questo sì mostra tutto il contenuto

Con `r+` puoi riscrivere il contenuto senza cancellare il resto del file — ma se il testo nuovo è più corto di quello che sostituisce, avanzano caratteri vecchi alla fine. Per questo esiste `truncate()`: taglia il file esattamente nella posizione in cui si trova il cursore.

In [ ]:
with open('reporte.txt', 'r+') as archivo:
    archivo.seek(0, 0)
    archivo.write("Riga 1\n")
    archivo.write("Riga 2\n")
    archivo.truncate()   # cancella qualsiasi resto di contenuto vecchio dopo questo punto
    archivo.seek(0, 0)
    print(archivo.read())

## Riepilogo: cosa porti a casa da questo capitolo

- `open(percorso, modalità)` apre un file. Modalità di base: `'r'` (leggere), `'w'` (scrivere, sovrascrive tutto) e `'a'` (aggiungere, conserva l'esistente).
- Usa sempre `with open(...) as archivo:` invece di `open()` + `close()` manuale. Garantisce la chiusura anche in caso di eccezione.
- `read()` prende tutto come una stringa (o i successivi `n` caratteri). Usalo con file piccoli.
- `readline()` legge una riga alla volta; combinato con un ciclo, percorre l'intero file senza caricarlo tutto in memoria.
- `readlines()` restituisce tutte le righe come lista, utile per l'accesso indicizzato.
- `write()` **non** aggiunge automaticamente gli a capo — bisogna includere `\n`.
- Copiare/filtrare tra file: apri l'origine in `'r'` e la destinazione in `'w'`/`'a'`, e scorri riga per riga.
- `tell()` dice dove si trova il cursore; `seek(offset, da_dove)` lo sposta. Le modalità combinate (`r+`, `w+`, `a+`) permettono di leggere e scrivere lo stesso file aperto, e `truncate()` taglia ciò che avanza dopo la posizione attuale.